In [ ]:
import json, urllib.request
from pathlib import Path

DATASETS = {
    "Panda": "OliverHausdoerfer/demo_src_pick_place_task_D0_robot_Panda_gripper_PandaGripper",
    "UR5e": "OliverHausdoerfer/demo_src_pick_place_task_D0_robot_UR5e_gripper_Robotiq85Gripper",
}

def read_public_file(repo, name):
    url = f"https://huggingface.co/datasets/{repo}/raw/main/meta/{name}"
    with urllib.request.urlopen(url, timeout=45) as response:
        return response.read().decode("utf-8")

for robot, repo in DATASETS.items():
    info = json.loads(read_public_file(repo, "info.json"))
    print("\nROBOT", robot, "repo", repo)
    print("episodes", info["total_episodes"], "frames", info["total_frames"], "fps", info["fps"])
    for key in ("action", "observation.state", "observation.eef_gripper", "observation.joint_gripper", "observation.images.agentview", "observation.images.wrist"):
        item = info["features"].get(key)
        print(key, None if item is None else (item["dtype"], item["shape"]))
    try:
        print("task", read_public_file(repo, "tasks.jsonl").splitlines()[0])
    except Exception as exc:
        print("task metadata unavailable:", type(exc).__name__, str(exc)[:120])


In [ ]:
import io
import importlib.util
print("pandas:", bool(importlib.util.find_spec("pandas")), "pyarrow:", bool(importlib.util.find_spec("pyarrow")))
if importlib.util.find_spec("pyarrow"):
    import pandas as pd
    for robot, repo in DATASETS.items():
        url = f"https://huggingface.co/datasets/{repo}/resolve/main/meta/tasks.parquet"
        payload = urllib.request.urlopen(url, timeout=45).read()
        table = pd.read_parquet(io.BytesIO(payload))
        print(robot, "tasks:", table.to_dict(orient="records"))


In [ ]:
import io, h5py, urllib.request
for robot, repo in DATASETS.items():
    url = f"https://huggingface.co/datasets/{repo}/resolve/main/eval_env_meta/demo.hdf5"
    raw = urllib.request.urlopen(url, timeout=60).read()
    with h5py.File(io.BytesIO(raw), "r") as f:
        print(robot, "groups", list(f.keys()), "attrs", list(f.attrs.keys()))
        for name, group in f.items():
            print(" ", name, "attrs", list(group.attrs.keys()), "children", list(group.keys())[:10])


In [ ]:
import io, json, numpy as np, pandas as pd, urllib.request
repo = DATASETS["Panda"]
base = f"https://huggingface.co/datasets/{repo}/resolve/main"
with h5py.File(io.BytesIO(urllib.request.urlopen(base + "/eval_env_meta/demo.hdf5").read()), "r") as f:
    meta = json.loads(f["data"].attrs["env_args"])
    print("environment:", meta.get("env_name"), "robot:", meta.get("env_kwargs", {}).get("robots"))
    print("environment keys:", list(meta.get("env_kwargs", {})))
raw = urllib.request.urlopen(base + "/data/chunk-000/file-000.parquet", timeout=180).read()
print("parquet bytes:", len(raw))
df = pd.read_parquet(io.BytesIO(raw))
print("rows:", len(df), "columns:", list(df.columns))
print("episode ids:", df["episode_index"].unique()[:10])
for key in ("action", "observation.state", "observation.eef", "observation.gripper", "observation.eef_gripper"):
    print(key, np.asarray(df[key].iloc[0]).round(4).tolist())
print("task indices:", df["task_index"].unique()[:10] if "task_index" in df else "missing")


In [ ]:
import json, urllib.request
for robot, repo in DATASETS.items():
    url = f"https://huggingface.co/datasets/{repo}/raw/main/meta/stats.json"
    stats = json.loads(urllib.request.urlopen(url).read())
    print(robot, "stats keys", list(stats)[:30])
    for key in ("observation.state", "observation.eef", "action"):
        item = stats.get(key)
        print(key, None if item is None else list(item), None if item is None else {k: len(v) if isinstance(v,list) else type(v).__name__ for k,v in item.items()})


In [ ]:
%pip install -q "lerobot[training,pi]"

In [ ]:
import importlib.metadata as md
from lerobot.datasets.lerobot_dataset import LeRobotDataset
print("lerobot version:", md.version("lerobot"))
ds = LeRobotDataset(DATASETS["Panda"], episodes=[0])
print("length:", len(ds), "features:", list(ds.meta.features))
sample = ds[0]
for key in ("task", "observation.state", "observation.eef", "action", "observation.images.agentview", "observation.images.wrist"):
    value = sample.get(key)
    print(key, getattr(value, "shape", value))


In [ ]:
import json, urllib.request
from lerobot.processor.rename_processor import rename_batch_keys, rename_stats
rename = {"observation.eef": "observation.state"}
mapped = rename_batch_keys(sample, rename)
print("Panda mapped sample state:", mapped["observation.state"].shape, mapped["observation.state"][:3])
for robot, repo in DATASETS.items():
    url = f"https://huggingface.co/datasets/{repo}/raw/main/meta/stats.json"
    stats = json.loads(urllib.request.urlopen(url).read())
    mapped_stats = rename_stats(stats, rename)
    print(robot, "mapped q01 length:", len(mapped_stats["observation.state"]["q01"]))


In [ ]:
ur = LeRobotDataset(DATASETS["UR5e"], episodes=[0])
ur_sample = ur[0]
ur_mapped = rename_batch_keys(ur_sample, rename)
print("UR5e frames:", len(ur))
print("task:", ur_sample.get("task"))
for key in ("observation.state", "action", "observation.images.agentview", "observation.images.wrist"):
    value = ur_mapped[key]
    print(key, getattr(value, "shape", value))
print("state first:", ur_mapped["observation.state"][:3])


In [ ]:
import inspect
import lerobot.processor.rename_processor as rp
print(inspect.signature(rp.rename_stats))
print(inspect.getsource(rp.rename_stats)[:1600])


In [ ]:
!lerobot-train --help | grep -E "rename_map|dataset.episodes|policy.pretrained_path|empty_cameras|train_expert_only|policy.n_action_steps|policy.dtype|steps|batch_size" | head -50

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
%pip install -q "lerobot[training,pi]"

In [ ]:
import lerobot, torch
from lerobot.policies.pi05.configuration_pi05 import PI05Config
print("lerobot", lerobot.__version__, "torch", torch.__version__)
print("GPU", torch.cuda.get_device_name(0), "VRAM GiB", round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print("pi05 defaults", {k:getattr(PI05Config(),k) for k in ("empty_cameras","n_action_steps","train_expert_only","freeze_vision_encoder","gradient_checkpointing","use_peft")})

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
for name in ("config.json", "policy_preprocessor.json", "policy_postprocessor.json"):
    try:
        path=hf_hub_download("lerobot/pi05_base", name)
        print(name, Path(path).stat().st_size, "bytes")
    except Exception as exc:
        print(name, type(exc).__name__, str(exc).splitlines()[0][:200])

In [ ]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download
config=json.loads(Path(hf_hub_download("lerobot/pi05_base","config.json")).read_text())
pre=json.loads(Path(hf_hub_download("lerobot/pi05_base","policy_preprocessor.json")).read_text())
print("config", {k:config.get(k) for k in ("type","input_features","output_features","empty_cameras","n_action_steps","normalization_mapping")})
print("preprocessor", pre)

In [ ]:
from huggingface_hub import hf_hub_download
try:
    p=hf_hub_download("google/paligemma-3b-pt-224","tokenizer_config.json")
    print("PaliGemma tokenizer access OK", p)
except Exception as exc:
    print("PaliGemma tokenizer blocked", type(exc).__name__, str(exc).splitlines()[0][:200])

In [ ]:
import inspect
from lerobot.scripts import lerobot_train
source=inspect.getsource(lerobot_train.train).splitlines()
for i,line in enumerate(source):
    if any(x in line for x in ("rename_map","make_policy","make_dataset","make_pre_post_processors","features")):
        print(f"{i+1}: {line.strip()}")

In [ ]:
for a,b in ((85,110),(127,165),(175,192)):
    print("---")
    for i in range(a-1,min(b,len(source))):
        print(f"{i+1}: {source[i]}")

In [ ]:
import json, pandas as pd
from huggingface_hub import hf_hub_download
DATASETS={"Panda":"OliverHausdoerfer/demo_src_pick_place_task_D0_robot_Panda_gripper_PandaGripper","UR5e":"OliverHausdoerfer/demo_src_pick_place_task_D0_robot_UR5e_gripper_Robotiq85Gripper"}
for robot,repo in DATASETS.items():
    info=json.loads(open(hf_hub_download(repo,"meta/info.json",repo_type="dataset")).read())
    print(robot,"episodes_path",info.get("episodes_path"))
    ep_path=info.get("episodes_path","meta/episodes/chunk-{episode_chunk:03d}/file-{episode_file:03d}.parquet").format(episode_chunk=0,episode_file=0)
    try:
        ep=pd.read_parquet(hf_hub_download(repo,ep_path,repo_type="dataset"))
        print("columns",ep.columns.tolist(),"first rows",ep.head(2).to_dict("records"))
    except Exception as exc:
        print(type(exc).__name__,str(exc).splitlines()[0][:150])

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
for robot,repo in DATASETS.items():
    ep=pd.read_parquet(hf_hub_download(repo,"meta/episodes/chunk-000/file-000.parquet",repo_type="dataset"),columns=["episode_index","length"])
    print(robot,"total episodes",len(ep),"first 20 frames",int(ep.length.iloc[:20].sum()),"first 40",int(ep.length.iloc[:40].sum()),"first 100",int(ep.length.iloc[:100].sum()),"mean frames",round(ep.length.mean(),1))

In [ ]:
# 본인 Hugging Face 토큰은 채팅에 보내지 말고 아래 셀을 직접 실행해 입력하세요.
# google/paligemma-3b-pt-224 사용 조건에 동의한 계정이어야 합니다.
!hf auth login

In [ ]:
import subprocess, importlib.metadata as md; from pathlib import Path; print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()); print("LeRobot:", md.version("lerobot")); print("Drive:", Path("/content/drive/MyDrive/VLA_experiments").exists()); print("smoke log:", Path("/content/panda_smoke.log").exists())

In [ ]:
from dataclasses import fields
from lerobot.configs.train import TrainPipelineConfig
print("TrainPipelineConfig relevant fields:")
for f in fields(TrainPipelineConfig):
    if any(t in f.name for t in ("eval","steps","workers","batch","rename","save","output","seed")):
        print(f.name,"default",f.default)
from lerobot.policies.factory import make_policy
print("make_policy signature", __import__("inspect").signature(make_policy))

In [ ]:
# 인증 셀을 실행한 뒤 이 짧은 학습 연결 시험을 실행합니다.
# 공개 Panda 시범 2판으로 딱 2번만 가중치를 업데이트합니다.
import subprocess
from pathlib import Path
smoke_dir=Path("/content/pi05_mimicgen_panda_smoke")
cmd=["lerobot-train",
     "--dataset.repo_id="+DATASETS["Panda"], "--dataset.episodes=[0,1]",
     "--policy.type=pi05", "--policy.pretrained_path=lerobot/pi05_base",
     "--policy.empty_cameras=1", "--policy.freeze_vision_encoder=true",
     "--policy.train_expert_only=true", "--policy.gradient_checkpointing=true",
     "--policy.dtype=bfloat16", "--policy.device=cuda", "--policy.push_to_hub=false",
     "--rename_map={\"observation.eef\":\"observation.state\"}",
     "--output_dir="+str(smoke_dir), "--batch_size=1", "--steps=2",
     "--num_workers=0", "--log_freq=1", "--save_checkpoint=false", "--seed=7"]
print("command:",cmd)
log=(smoke_dir.parent/"panda_smoke.log").open("w")
smoke_process=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT)
log.close()
print("pid:",smoke_process.pid, "log:",smoke_dir.parent/"panda_smoke.log")

In [ ]:
from pathlib import Path
import subprocess
print("returncode:", smoke_process.poll())
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.free", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
log_path=Path("/content/panda_smoke.log")
print("log tail:\n", log_path.read_text(errors="replace")[-2500:] if log_path.exists() else "missing")

In [ ]:
from pathlib import Path; import shutil; d=Path("/content/drive/MyDrive/VLA_experiments/Pi05_MimicGen_Panda_L4_smoke_20261123"); d.mkdir(parents=True, exist_ok=True); shutil.copy2("/content/panda_smoke.log", d/"panda_smoke.log"); print("saved", d, (d/"panda_smoke.log").stat().st_size)

In [ ]:
import subprocess
print(subprocess.run(["ps", "-o", "pid,stat,etime,rss,wchan:25,cmd", "-p", str(smoke_process.pid)], capture_output=True, text=True).stdout)
print(subprocess.run(["du", "-sh", "/root/.cache/huggingface/hub"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
# 성능 비교를 시작할 때 쓸 고정 분할 후보. 시범(episode) 단위로 나눠 사진 누출을 피합니다.
import numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
ids=np.random.default_rng(7).permutation(500)
train_ids=sorted(ids[:100].tolist())
validation_ids=sorted(ids[100:120].tolist())
heldout_ids=sorted(ids[120:140].tolist())
print("train/validation/heldout demos",len(train_ids),len(validation_ids),len(heldout_ids))
print("overlap",len(set(train_ids)&set(validation_ids)),len(set(train_ids)&set(heldout_ids)))
for robot,repo in DATASETS.items():
    ep=pd.read_parquet(hf_hub_download(repo,"meta/episodes/chunk-000/file-000.parquet",repo_type="dataset"),columns=["episode_index","length"]).set_index("episode_index")
    print(robot,"train frames",int(ep.loc[train_ids,"length"].sum()),"val frames",int(ep.loc[validation_ids,"length"].sum()))
print("주의: 동일한 시범 번호가 두 로봇에서 동일한 물리적 시작 위치를 뜻하는지는 확인되지 않았습니다.")

In [ ]:
from huggingface_hub import list_repo_files
files=list_repo_files("lerobot/pi05_base")
print("file count",len(files))
print("tokenizer files",[x for x in files if "tokenizer" in x.lower()][:30])
print("model files",[x for x in files if x.endswith(".safetensors")][:10])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
root=Path("/content/drive/MyDrive/VLA_experiments")
print("research Drive VLA_experiments exists",root.exists())
print("UR5e model path exists",(root/"UR5e_Pi05_40demo_1epoch_seed7/pretrained_model").exists())